# FoldPipe MD17 + SchNet rigorous benchmark

Ten paired, order-alternating passes on five revision-pinned private MD17 shards. The notebook emits raw traces, bootstrap intervals, a plot, a source manifest, and a Markdown report.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch-geometric",
        "huggingface_hub",
        "matplotlib",
        "psutil",
        "biopython",
        "google-api-python-client",
        "google-auth-oauthlib",
    ],
    check=True,
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("The benchmark requires the requested NVIDIA T4 GPU")

torch_version = torch.__version__.split("+")[0]
cuda_tag = torch.version.cuda.replace(".", "")
pyg_wheels = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_tag}.html"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyg_lib",
        "torch_scatter",
        "torch_sparse",
        "torch_cluster",
        "torch_spline_conv",
        "-f",
        pyg_wheels,
    ],
    check=True,
)

print({
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
})


In [ ]:
import base64
import json
import os
from pathlib import Path

WORK_ROOT = Path("/kaggle/working/foldpipe-benchmark")
PAYLOADS = {"foldpipe/__init__.py": "ZnJvbSAubG9hZGVyIGltcG9ydCBBc3luY0ZvbGRQaXBlTG9hZGVyCmZyb20gLnByaW9uX2xvYWRlciBpbXBvcnQgUHJpb25TdHJlYW1lcgoKX19hbGxfXyA9IFsiQXN5bmNGb2xkUGlwZUxvYWRlciIsICJQcmlvblN0cmVhbWVyIl0K", "foldpipe/loader.py": "aW1wb3J0IGNvbmN1cnJlbnQuZnV0dXJlcwoKY2xhc3MgQXN5bmNGb2xkUGlwZUxvYWRlcjoKICAgICIiIgogICAgVHJ1ZSBPKDEpIGJvdW5kZWQtbWVtb3J5IGFzeW5jaHJvbm91cyBzdHJlYW1pbmcgZGF0YWxvYWRlci4KICAgIERvd25sb2FkcyBuYXRpdmUgUHlUb3JjaCAucHQgY2h1bmsgZmlsZXMgdmlhIGEgZ2VuZXJpYyBTb3VyY2UgYmFja2VuZCBpbiB0aGUgYmFja2dyb3VuZC4KICAgIEhpZGVzIG5ldHdvcmsgSS9PIGxhdGVuY3kgYmVoaW5kIEdQVSBjb21wdXRhdGlvbi4KICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNvdXJjZSwgYmF0Y2hfc2l6ZT0xMjgsIGJhdGNoX2ZuPU5vbmUpOgogICAgICAgIHNlbGYuc291cmNlID0gc291cmNlCiAgICAgICAgc2VsZi5iYXRjaF9zaXplID0gYmF0Y2hfc2l6ZQogICAgICAgIHNlbGYuYmF0Y2hfZm4gPSBiYXRjaF9mbiBvciBzZWxmLl9kZWZhdWx0X2JhdGNoX2ZuCgogICAgZGVmIF9kZWZhdWx0X2JhdGNoX2ZuKHNlbGYsIGNodW5rKToKICAgICAgICAiIiJEZWZhdWx0IHRlbnNvciBzbGljaW5nIGZvciBzeW1tZXRyaWMgdGVuc29ycy4iIiIKICAgICAgICBmb3IgYiBpbiByYW5nZSgwLCBjaHVuay5zaXplKDApLCBzZWxmLmJhdGNoX3NpemUpOgogICAgICAgICAgICB5aWVsZCBjaHVua1tiOmIrc2VsZi5iYXRjaF9zaXplXQoKICAgIGRlZiBfX2l0ZXJfXyhzZWxmKToKICAgICAgICAiIiJDb25zdW1lciBwaXBlbGluZS4iIiIKICAgICAgICBmaWxlX2l0ZXJhdG9yID0gc2VsZi5zb3VyY2UuaXRlcl9maWxlcygpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBmaXJzdF9maWxlID0gbmV4dChmaWxlX2l0ZXJhdG9yKQogICAgICAgIGV4Y2VwdCBTdG9wSXRlcmF0aW9uOgogICAgICAgICAgICByZXR1cm4KCiAgICAgICAgd2l0aCBjb25jdXJyZW50LmZ1dHVyZXMuVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPTEpIGFzIGV4ZWN1dG9yOgogICAgICAgICAgICAjIEtpY2sgb2ZmIHRoZSBwcmVmZXRjaCBmb3IgQ2h1bmsgMAogICAgICAgICAgICBmdXR1cmVfY2h1bmsgPSBleGVjdXRvci5zdWJtaXQoc2VsZi5zb3VyY2UuZG93bmxvYWRfY2h1bmssIGZpcnN0X2ZpbGUpCiAgICAgICAgICAgIAogICAgICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICAgICAgIyBCbG9jayBvbmx5IGlmIEdQVSBpcyBmYXN0ZXIgdGhhbiBuZXR3b3JrCiAgICAgICAgICAgICAgICBjaHVua190ZW5zb3IgPSBmdXR1cmVfY2h1bmsucmVzdWx0KCkKICAgICAgICAgICAgICAgIAogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIG5leHRfZmlsZSA9IG5leHQoZmlsZV9pdGVyYXRvcikKICAgICAgICAgICAgICAgICAgICAjIEtpY2sgb2ZmIHByZWZldGNoIGZvciBDaHVuayBOKzEgaW1tZWRpYXRlbHkKICAgICAgICAgICAgICAgICAgICBmdXR1cmVfY2h1bmsgPSBleGVjdXRvci5zdWJtaXQoc2VsZi5zb3VyY2UuZG93bmxvYWRfY2h1bmssIG5leHRfZmlsZSkKICAgICAgICAgICAgICAgICAgICBoYXNfbmV4dCA9IFRydWUKICAgICAgICAgICAgICAgIGV4Y2VwdCBTdG9wSXRlcmF0aW9uOgogICAgICAgICAgICAgICAgICAgIGhhc19uZXh0ID0gRmFsc2UKICAgICAgICAgICAgICAgIAogICAgICAgICAgICAgICAgIyBZaWVsZCBiYXRjaGVzIHRvIHRoZSBHUFUKICAgICAgICAgICAgICAgIGZvciBiYXRjaCBpbiBzZWxmLmJhdGNoX2ZuKGNodW5rX3RlbnNvcik6CiAgICAgICAgICAgICAgICAgICAgeWllbGQgYmF0Y2gKICAgICAgICAgICAgICAgIAogICAgICAgICAgICAgICAgIyBFeHBsaWNpdGx5IGRlbGV0ZSB0aGUgY2h1bmsgKHJlbHlpbmcgb24gUHl0aG9uIEdDIGZvciBPKDEpIGJvdW5kKQogICAgICAgICAgICAgICAgZGVsIGNodW5rX3RlbnNvcgogICAgICAgICAgICAgICAgCiAgICAgICAgICAgICAgICBpZiBub3QgaGFzX25leHQ6CiAgICAgICAgICAgICAgICAgICAgYnJlYWsK", "foldpipe/prion_loader.py": "aW1wb3J0IG9zCmltcG9ydCB0b3JjaAoKZnJvbSB0b3JjaF9nZW9tZXRyaWMuZGF0YSBpbXBvcnQgRGF0YQpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IEl0ZXJhYmxlRGF0YXNldApmcm9tIEJpbyBpbXBvcnQgUERCCmltcG9ydCBnbG9iCgpFTEVNRU5UX1RPX1ogPSB7CiAgICAnSCc6IDEsICdDJzogNiwgJ04nOiA3LCAnTyc6IDgsICdTJzogMTYsICdQJzogMTUsCn0KCmNsYXNzIFByaW9uU3RyZWFtZXIoSXRlcmFibGVEYXRhc2V0KToKICAgICIiIgogICAgVHJ1ZSBPKDEpIHN0cmVhbWluZyBwYXJzZXIgZm9yIHJhdyBQREIgZmlsZXMuIAogICAgRG9lcyBOT1QgdXNlIFB5RyBJbk1lbW9yeURhdGFzZXQgdG8gYXZvaWQgYnVpbGRpbmcgYSBkYXRhX2xpc3QuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCByYXdfZGlyKToKICAgICAgICBzZWxmLnJhd19kaXIgPSByYXdfZGlyCiAgICAgICAgc2VsZi5yYXdfZmlsZXMgPSBnbG9iLmdsb2Iob3MucGF0aC5qb2luKHNlbGYucmF3X2RpciwgJyoucGRiJykpCiAgICAgICAgCiAgICBkZWYgX19pdGVyX18oc2VsZik6CiAgICAgICAgaWYgbm90IHNlbGYucmF3X2ZpbGVzOgogICAgICAgICAgICBwcmludChmIk5vIFBEQiBmaWxlcyBmb3VuZCBpbiB7c2VsZi5yYXdfZGlyfS4iKQogICAgICAgICAgICByZXR1cm4KCiAgICAgICAgcGFyc2VyID0gUERCLlBEQlBhcnNlcihRVUlFVD1UcnVlKQogICAgICAgIAogICAgICAgIGZvciByYXdfcGF0aCBpbiBzZWxmLnJhd19maWxlczoKICAgICAgICAgICAgc3RydWN0dXJlID0gcGFyc2VyLmdldF9zdHJ1Y3R1cmUoInByaW9uIiwgcmF3X3BhdGgpCiAgICAgICAgICAgIHpfbGlzdCA9IFtdCiAgICAgICAgICAgIHBvc19saXN0ID0gW10KICAgICAgICAgICAgCiAgICAgICAgICAgIGZvciBtb2RlbCBpbiBzdHJ1Y3R1cmU6CiAgICAgICAgICAgICAgICBmb3IgY2hhaW4gaW4gbW9kZWw6CiAgICAgICAgICAgICAgICAgICAgZm9yIHJlc2lkdWUgaW4gY2hhaW46CiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBhdG9tIGluIHJlc2lkdWU6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbGVtZW50ID0gYXRvbS5lbGVtZW50LnN0cmlwKCkudXBwZXIoKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgel9saXN0LmFwcGVuZChFTEVNRU5UX1RPX1ouZ2V0KGVsZW1lbnQsIDApKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcG9zX2xpc3QuYXBwZW5kKGF0b20uY29vcmQpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgCiAgICAgICAgICAgIHpfdGVuc29yID0gdG9yY2gudGVuc29yKHpfbGlzdCwgZHR5cGU9dG9yY2gubG9uZykKICAgICAgICAgICAgcG9zX3RlbnNvciA9IHRvcmNoLnRlbnNvcihwb3NfbGlzdCwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICAgICAgCiAgICAgICAgICAgIHlpZWxkIERhdGEoej16X3RlbnNvciwgcG9zPXBvc190ZW5zb3IpCg==", "foldpipe/sources.py": "aW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmltcG9ydCB0b3JjaAppbXBvcnQgcmVxdWVzdHMKZnJvbSB1cmxsaWIucGFyc2UgaW1wb3J0IHF1b3RlCmZyb20gYWJjIGltcG9ydCBBQkMsIGFic3RyYWN0bWV0aG9kCmZyb20gZ29vZ2xlYXBpY2xpZW50LmRpc2NvdmVyeSBpbXBvcnQgYnVpbGQKZnJvbSBnb29nbGVhcGljbGllbnQuaHR0cCBpbXBvcnQgTWVkaWFJb0Jhc2VEb3dubG9hZApmcm9tIGdvb2dsZS5vYXV0aDIuY3JlZGVudGlhbHMgaW1wb3J0IENyZWRlbnRpYWxzCmZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkZpbGVTeXN0ZW0KCmNsYXNzIFNvdXJjZShBQkMpOgogICAgQGFic3RyYWN0bWV0aG9kCiAgICBkZWYgaXRlcl9maWxlcyhzZWxmKToKICAgICAgICAiIiJZaWVsZHMgaWRlbnRpZmllcnMgZm9yIHRoZSBjaHVuayBmaWxlcyBsYXppbHkgdG8gc3RyaWN0bHkgYm91bmQgbWVtb3J5LiIiIgogICAgICAgIHBhc3MKCiAgICBAYWJzdHJhY3RtZXRob2QKICAgIGRlZiBkb3dubG9hZF9jaHVuayhzZWxmLCBpZGVudGlmaWVyKToKICAgICAgICAiIiJEb3dubG9hZHMgYSBjaHVuayBhbmQgcmV0dXJucyBhIFB5VG9yY2ggdGVuc29yIGRpcmVjdGx5IGluIG1lbW9yeS4iIiIKICAgICAgICBwYXNzCgoKY2xhc3MgR29vZ2xlRHJpdmVTb3VyY2UoU291cmNlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBmb2xkZXJfaWQsIGNyZWRlbnRpYWxzX2pzb24pOgogICAgICAgIHNlbGYuZm9sZGVyX2lkID0gZm9sZGVyX2lkCiAgICAgICAgCiAgICAgICAgaWYgaXNpbnN0YW5jZShjcmVkZW50aWFsc19qc29uLCBzdHIpOgogICAgICAgICAgICB3aXRoIG9wZW4oY3JlZGVudGlhbHNfanNvbiwgJ3InKSBhcyBmOgogICAgICAgICAgICAgICAgc2VsZi5jcmVkc19kaWN0ID0ganNvbi5sb2FkKGYpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5jcmVkc19kaWN0ID0gY3JlZGVudGlhbHNfanNvbgogICAgICAgICAgICAKICAgICAgICAjIE5hcnJvd3Mgc2NvcGUgdG8gcmVhZG9ubHkgYXMgcmVxdWVzdGVkIGJ5IHBlZXIgcmV2aWV3CiAgICAgICAgY3JlZHMgPSBDcmVkZW50aWFscy5mcm9tX2F1dGhvcml6ZWRfdXNlcl9pbmZvKHNlbGYuY3JlZHNfZGljdCwgc2NvcGVzPVsnaHR0cHM6Ly93d3cuZ29vZ2xlYXBpcy5jb20vYXV0aC9kcml2ZS5yZWFkb25seSddKQogICAgICAgIHNlbGYuZHJpdmVfc2VydmljZSA9IGJ1aWxkKCdkcml2ZScsICd2MycsIGNyZWRlbnRpYWxzPWNyZWRzKQoKICAgIGRlZiBpdGVyX2ZpbGVzKHNlbGYpOgogICAgICAgIHF1ZXJ5ID0gZiIne3NlbGYuZm9sZGVyX2lkfScgaW4gcGFyZW50cyBhbmQgbmFtZSBjb250YWlucyAnY2hlY2twb2ludF9iYXRjaF8nIGFuZCB0cmFzaGVkID0gZmFsc2UiCiAgICAgICAgcGFnZV90b2tlbiA9IE5vbmUKICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICByZXN1bHRzID0gc2VsZi5kcml2ZV9zZXJ2aWNlLmZpbGVzKCkubGlzdCgKICAgICAgICAgICAgICAgIHE9cXVlcnksIAogICAgICAgICAgICAgICAgZmllbGRzPSJuZXh0UGFnZVRva2VuLCBmaWxlcyhpZCwgbmFtZSkiLCAKICAgICAgICAgICAgICAgIG9yZGVyQnk9Im5hbWVfbmF0dXJhbCIsCiAgICAgICAgICAgICAgICBwYWdlVG9rZW49cGFnZV90b2tlbgogICAgICAgICAgICApLmV4ZWN1dGUoKQogICAgICAgICAgICAKICAgICAgICAgICAgZm9yIGZpbGVfaW5mbyBpbiByZXN1bHRzLmdldCgnZmlsZXMnLCBbXSk6CiAgICAgICAgICAgICAgICB5aWVsZCBmaWxlX2luZm8KICAgICAgICAgICAgICAgIAogICAgICAgICAgICBwYWdlX3Rva2VuID0gcmVzdWx0cy5nZXQoJ25leHRQYWdlVG9rZW4nKQogICAgICAgICAgICBpZiBub3QgcGFnZV90b2tlbjoKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgZGVmIGRvd25sb2FkX2NodW5rKHNlbGYsIGlkZW50aWZpZXIpOgogICAgICAgIGZpbGVfaWQgPSBpZGVudGlmaWVyWydpZCddCiAgICAgICAgcmVxdWVzdCA9IHNlbGYuZHJpdmVfc2VydmljZS5maWxlcygpLmdldF9tZWRpYShmaWxlSWQ9ZmlsZV9pZCkKICAgICAgICBmaCA9IGlvLkJ5dGVzSU8oKQogICAgICAgIGRvd25sb2FkZXIgPSBNZWRpYUlvQmFzZURvd25sb2FkKGZoLCByZXF1ZXN0KQogICAgICAgIGRvbmUgPSBGYWxzZQogICAgICAgIHdoaWxlIGRvbmUgaXMgRmFsc2U6CiAgICAgICAgICAgIHN0YXR1cywgZG9uZSA9IGRvd25sb2FkZXIubmV4dF9jaHVuaygpCiAgICAgICAgZmguc2VlaygwKQogICAgICAgIHJldHVybiB0b3JjaC5sb2FkKGZoLCBtYXBfbG9jYXRpb249J2NwdScsIHdlaWdodHNfb25seT1GYWxzZSkKCgpmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgSGZGaWxlU3lzdGVtLCBIZkFwaQoKY2xhc3MgSHVnZ2luZ0ZhY2VTb3VyY2UoU291cmNlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHJlcG9faWQsCiAgICAgICAgZm9sZGVyX3BhdGg9IiIsCiAgICAgICAgdG9rZW49Tm9uZSwKICAgICAgICByZXZpc2lvbj1Ob25lLAogICAgICAgIHRyYW5zZmVyX29ic2VydmVyPU5vbmUsCiAgICApOgogICAgICAgIHNlbGYucmVwb19pZCA9IHJlcG9faWQKICAgICAgICBzZWxmLmZvbGRlcl9wYXRoID0gZm9sZGVyX3BhdGguc3RyaXAoIi8iKQogICAgICAgIHNlbGYudG9rZW4gPSB0b2tlbgogICAgICAgIHNlbGYucmV2aXNpb24gPSByZXZpc2lvbgogICAgICAgIHNlbGYudHJhbnNmZXJfb2JzZXJ2ZXIgPSB0cmFuc2Zlcl9vYnNlcnZlcgogICAgICAgIHNlbGYuZnMgPSBIZkZpbGVTeXN0ZW0odG9rZW49dG9rZW4pCiAgICAgICAgc2VsZi5hcGkgPSBIZkFwaSh0b2tlbj10b2tlbikKCiAgICBkZWYgaXRlcl9maWxlcyhzZWxmKToKICAgICAgICAjIEdlbmVyYXRlIGl0ZW1zIGNvbXBsZXRlbHkgbGF6aWx5IGRpcmVjdGx5IGZyb20gdGhlIEh1Z2dpbmcgRmFjZSBBUEkKICAgICAgICBwYXRoX2luX3JlcG8gPSBzZWxmLmZvbGRlcl9wYXRoIGlmIHNlbGYuZm9sZGVyX3BhdGggZWxzZSBOb25lCiAgICAgICAgdHJlZV9nZW5lcmF0b3IgPSBzZWxmLmFwaS5saXN0X3JlcG9fdHJlZSgKICAgICAgICAgICAgcmVwb19pZD1zZWxmLnJlcG9faWQsCiAgICAgICAgICAgIHJlcG9fdHlwZT0iZGF0YXNldCIsCiAgICAgICAgICAgIHBhdGhfaW5fcmVwbz1wYXRoX2luX3JlcG8sCiAgICAgICAgICAgIHJldmlzaW9uPXNlbGYucmV2aXNpb24sCiAgICAgICAgICAgIHJlY3Vyc2l2ZT1GYWxzZSwKICAgICAgICAgICAgZXhwYW5kPUZhbHNlCiAgICAgICAgKQogICAgICAgIGZvciBpdGVtIGluIHRyZWVfZ2VuZXJhdG9yOgogICAgICAgICAgICBpZiAiY2hlY2twb2ludF9iYXRjaF8iIGluIGl0ZW0ucmZpbGVuYW1lOgogICAgICAgICAgICAgICAgeWllbGQgaXRlbS5yZmlsZW5hbWUKCiAgICBkZWYgZG93bmxvYWRfY2h1bmsoc2VsZiwgaWRlbnRpZmllcik6CiAgICAgICAgZmlsZV9wYXRoID0gaWRlbnRpZmllcgogICAgICAgICMgV2Ugc3RyaXAgdGhlIGxlYWRpbmcgImRhdGFzZXRzLyIgcHJlZml4IGZyb20gSGZGaWxlU3lzdGVtIGlmIGl0IGV4aXN0cwogICAgICAgIGlmIGZpbGVfcGF0aC5zdGFydHN3aXRoKGYiZGF0YXNldHMve3NlbGYucmVwb19pZH0vIik6CiAgICAgICAgICAgIGZpbGVfcGF0aCA9IGZpbGVfcGF0aFtsZW4oZiJkYXRhc2V0cy97c2VsZi5yZXBvX2lkfS8iKTpdCiAgICAgICAgICAgIAogICAgICAgIHJldmlzaW9uID0gcXVvdGUoc2VsZi5yZXZpc2lvbiBvciAibWFpbiIsIHNhZmU9IiIpCiAgICAgICAgdXJsID0gZiJodHRwczovL2h1Z2dpbmdmYWNlLmNvL2RhdGFzZXRzL3tzZWxmLnJlcG9faWR9L3Jlc29sdmUve3JldmlzaW9ufS97ZmlsZV9wYXRofSIKICAgICAgICBoZWFkZXJzID0ge30KICAgICAgICBpZiBzZWxmLnRva2VuOgogICAgICAgICAgICBoZWFkZXJzWyJBdXRob3JpemF0aW9uIl0gPSBmIkJlYXJlciB7c2VsZi50b2tlbn0iCgogICAgICAgIGV2ZW50ID0gewogICAgICAgICAgICAiaWRlbnRpZmllciI6IGlkZW50aWZpZXIsCiAgICAgICAgICAgICJkb3dubG9hZF9zdGFydCI6IHRpbWUucGVyZl9jb3VudGVyKCksCiAgICAgICAgICAgICJkb3dubG9hZF9maW5pc2giOiBOb25lLAogICAgICAgICAgICAiZGVzZXJpYWxpemVfZmluaXNoIjogTm9uZSwKICAgICAgICAgICAgImJ5dGVzX2Rvd25sb2FkZWQiOiAwLAogICAgICAgIH0KCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIFN0cmVhbSBkaXJlY3RseSB0byBSQU0gdmlhIHJlcXVlc3RzLCBhdm9pZGluZyBsb2NhbCBkaXNrIGNhY2hlIGFuZAogICAgICAgICAgICAjIGRvdWJsZS1tYXRlcmlhbGl6YXRpb24uIENvdW50IHBheWxvYWQgYnl0ZXMgZm9yIGJlbmNobWFyayB0cmFjaW5nLgogICAgICAgICAgICByZXNwb25zZSA9IHJlcXVlc3RzLmdldCh1cmwsIGhlYWRlcnM9aGVhZGVycywgc3RyZWFtPVRydWUpCiAgICAgICAgICAgIHJlc3BvbnNlLnJhaXNlX2Zvcl9zdGF0dXMoKQoKICAgICAgICAgICAgZmggPSBpby5CeXRlc0lPKCkKICAgICAgICAgICAgZm9yIGJsb2NrIGluIHJlc3BvbnNlLml0ZXJfY29udGVudChjaHVua19zaXplPTEwMjQgKiAxMDI0KToKICAgICAgICAgICAgICAgIGlmIGJsb2NrOgogICAgICAgICAgICAgICAgICAgIGV2ZW50WyJieXRlc19kb3dubG9hZGVkIl0gKz0gbGVuKGJsb2NrKQogICAgICAgICAgICAgICAgICAgIGZoLndyaXRlKGJsb2NrKQoKICAgICAgICAgICAgZXZlbnRbImRvd25sb2FkX2ZpbmlzaCJdID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICBmaC5zZWVrKDApCiAgICAgICAgICAgIGNodW5rID0gdG9yY2gubG9hZChmaCwgbWFwX2xvY2F0aW9uPSdjcHUnLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgICAgIGV2ZW50WyJkZXNlcmlhbGl6ZV9maW5pc2giXSA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgcmV0dXJuIGNodW5rCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgIGV2ZW50WyJlcnJvciJdID0gdHlwZShleGMpLl9fbmFtZV9fCiAgICAgICAgICAgIHJhaXNlCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgaWYgc2VsZi50cmFuc2Zlcl9vYnNlcnZlciBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHNlbGYudHJhbnNmZXJfb2JzZXJ2ZXIoZXZlbnQuY29weSgpKQoKY2xhc3MgU3ludGhldGljTGF0ZW5jeVNvdXJjZShTb3VyY2UpOgogICAgIiIiCiAgICBBIGNvbnRyb2xsZWQgc291cmNlIHRoYXQgYWxsb3dzIGNvbmZpZ3VyaW5nIGV4YWN0IGFydGlmaWNpYWwgbmV0d29yayBsYXRlbmN5CiAgICBhbmQgcmV0dXJucyBmaXhlZC1zaXplIGR1bW15IHRlbnNvcnMuIENydWNpYWwgZm9yIGlzb2xhdGVkIG1lY2hhbmlzbSBleHBlcmltZW50cy4KICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9jaHVua3M9MTUsIGxhdGVuY3lfbXM9MTAwLCBjaHVua19zaXplPTIwMDAwKToKICAgICAgICBzZWxmLm51bV9jaHVua3MgPSBudW1fY2h1bmtzCiAgICAgICAgc2VsZi5sYXRlbmN5X21zID0gbGF0ZW5jeV9tcwogICAgICAgIHNlbGYuY2h1bmtfc2l6ZSA9IGNodW5rX3NpemUKICAgICAgICAKICAgIGRlZiBpdGVyX2ZpbGVzKHNlbGYpOgogICAgICAgIGZvciBpIGluIHJhbmdlKHNlbGYubnVtX2NodW5rcyk6CiAgICAgICAgICAgIHlpZWxkIGYic3ludGhldGljX2NodW5rX3tpfS5wdCIKICAgICAgICAgICAgCiAgICBkZWYgZG93bmxvYWRfY2h1bmsoc2VsZiwgaWRlbnRpZmllcik6CiAgICAgICAgaWYgc2VsZi5sYXRlbmN5X21zID4gMDoKICAgICAgICAgICAgdGltZS5zbGVlcChzZWxmLmxhdGVuY3lfbXMgLyAxMDAwLjApCiAgICAgICAgcmV0dXJuIHRvcmNoLnJhbmRuKHNlbGYuY2h1bmtfc2l6ZSwgMykKCmNsYXNzIFByZWVudW1lcmF0ZWRTb3VyY2UoU291cmNlKToKICAgICIiIgogICAgQSB3cmFwcGVyIHNvdXJjZSB0aGF0IHRha2VzIGEgcHJlLWZldGNoZWQgbGlzdCBvZiBmaWxlIGlkZW50aWZpZXJzIGFuZCBhbiB1bmRlcmx5aW5nIHNvdXJjZS4KICAgIFRoaXMgZWxpbWluYXRlcyByZW1vdGUgQVBJIGRpc2NvdmVyeSBvdmVyaGVhZCAoZS5nLiBsaXN0X3JlcG9fdHJlZSkgZHVyaW5nIGJlbmNobWFya3MsCiAgICBlbnN1cmluZyB0aGF0IHRpbWluZyBzdHJpY3RseSBtZWFzdXJlcyB0aGUgb3JjaGVzdHJhdGlvbiBwaXBlbGluZS4KICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlkZW50aWZpZXJzLCB1bmRlcmx5aW5nX3NvdXJjZSk6CiAgICAgICAgc2VsZi5pZGVudGlmaWVycyA9IGlkZW50aWZpZXJzCiAgICAgICAgc2VsZi51bmRlcmx5aW5nX3NvdXJjZSA9IHVuZGVybHlpbmdfc291cmNlCgogICAgZGVmIGl0ZXJfZmlsZXMoc2VsZik6CiAgICAgICAgZm9yIGlkZW50aWZpZXIgaW4gc2VsZi5pZGVudGlmaWVyczoKICAgICAgICAgICAgeWllbGQgaWRlbnRpZmllcgoKICAgIGRlZiBkb3dubG9hZF9jaHVuayhzZWxmLCBpZGVudGlmaWVyKToKICAgICAgICByZXR1cm4gc2VsZi51bmRlcmx5aW5nX3NvdXJjZS5kb3dubG9hZF9jaHVuayhpZGVudGlmaWVyKQo=", "scripts/benchmark_molecular.py": "..."}
SOURCE_MANIFEST = json.loads("{\"base_git_commit\": \"f19735f10935fd44a3fbf4bfee0660adf83a111b\", \"bundle_sha256\": \"0ed7e4c583621edc793234e51dafc3756221f7e2337e1d86708c1d5f25fcad39\", \"files\": {\"foldpipe/__init__.py\": {\"bytes\": 132, \"sha256\": \"e3a8d8a89f6aa3c93b755028d0816e47a27889346a37ebc92c7a9b071fe11710\"}, \"foldpipe/loader.py\": {\"bytes\": 1965, \"sha256\": \"e3ae29bf11457cc11eb509d181a9fc1fde1089cce2c355e80c9a7915e02b711d\"}, \"foldpipe/prion_loader.py\": {\"bytes\": 1468, \"sha256\": \"49f8ec2932268d8d02f6acc4d645414a2411bfa7293405dc6c11a5e5033180b5\"}, \"foldpipe/sources.py\": {\"bytes\": 6527, \"sha256\": \"9e49b646a1d24eae9d7b0e2dd752e5cc1ad9fc9623dc24682ecbeb8eab213a37\"}, \"scripts/benchmark_molecular.py\": {\"bytes\": 21170, \"sha256\": \"7dcadaa67c52c82c79da461430a1a1622f8be90bcae70989b2522937d9252037\"}}, \"format_version\": 1, \"working_tree_dirty\": true}")

for relative_path, encoded_payload in PAYLOADS.items():
    destination = WORK_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded_payload))

manifest_path = WORK_ROOT / "source_manifest.json"
manifest_path.write_text(
    json.dumps(SOURCE_MANIFEST, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
os.chdir(WORK_ROOT)
print({
    "source_bundle_sha256": SOURCE_MANIFEST["bundle_sha256"],
    "base_git_commit": SOURCE_MANIFEST["base_git_commit"],
    "embedded_files": len(SOURCE_MANIFEST["files"]),
})


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

secret_token = UserSecretsClient().get_secret("HF_TOKEN")
run_environment = os.environ.copy()
run_environment.update({
    "HF_TOKEN": secret_token,
    "FOLDPIPE_HF_REPO_ID": "aviatorlf/md17-shards",
    "FOLDPIPE_MAX_CHUNKS": "5",
    "FOLDPIPE_NUM_RUNS": "10",
    "FOLDPIPE_BOOTSTRAP_SAMPLES": "20000",
    "FOLDPIPE_SOURCE_MANIFEST": str(Path.cwd() / "source_manifest.json"),
    "PYTHONUNBUFFERED": "1",
})
existing_pythonpath = run_environment.get("PYTHONPATH")
run_environment["PYTHONPATH"] = os.pathsep.join(
    item for item in (str(Path.cwd()), existing_pythonpath) if item
)

subprocess.run(
    [sys.executable, "scripts/benchmark_molecular.py"],
    env=run_environment,
    check=True,
)
del secret_token
run_environment.pop("HF_TOKEN", None)

results_path = Path("results/benchmark_stats_md17.json")
plot_path = Path("results/benchmark_comparison_md17.png")
results = json.loads(results_path.read_text(encoding="utf-8"))

seq = results["sequential"]
fold = results["foldpipe"]
effect = results["paired_effect"]
speed_ci = effect["speedup_ratio"]["ci_95"]
saved_ci = effect["time_saved_s"]["ci_95"]
ratio_includes_null = speed_ci["low"] <= 1.0 <= speed_ci["high"]
additive_includes_null = saved_ci["low"] <= 0.0 <= saved_ci["high"]
if ratio_includes_null and additive_includes_null:
    interpretation = "Both paired intervals include their no-effect values; this run is inconclusive about a speed advantage."
elif not ratio_includes_null and additive_includes_null:
    interpretation = (
        "The mean-ratio interval excludes 1 in FoldPipe's favor, but the additive time-saved interval includes 0. "
        "The estimands disagree under high run-to-run variability, so the result should not be presented as a universal speedup."
    )
elif ratio_includes_null and not additive_includes_null:
    interpretation = (
        "The additive time-saved interval excludes 0, but the mean-ratio interval includes 1. "
        "The estimands disagree, so both paired summaries and the raw runs should be reported."
    )
else:
    interpretation = "Both paired intervals exclude their no-effect values for this protocol."

report = f"""# FoldPipe MD17 + SchNet benchmark

- Generated: {results['metadata']['generated_at_utc']}
- Hardware: {results['metadata']['gpu']}
- Dataset: `{results['metadata']['dataset_repo']}@{results['metadata']['dataset_revision']}`
- Source bundle: `{results['metadata']['code']['source_bundle']['bundle_sha256']}`
- Base Git commit: `{results['metadata']['code']['source_bundle']['base_git_commit']}`
- Protocol: {results['metadata']['paired_runs']} paired, order-alternating passes; {results['metadata']['shards_per_pass']} pinned shards per pass; batch size {results['metadata']['batch_size']}
- Warm-up: {results['metadata']['warmup_protocol']}

| Metric | Sequential | FoldPipe |
| --- | ---: | ---: |
| Mean time (s) | {seq['time_s']['mean']:.3f} | {fold['time_s']['mean']:.3f} |
| 95% bootstrap CI, mean time (s) | [{seq['time_s']['ci_95']['low']:.3f}, {seq['time_s']['ci_95']['high']:.3f}] | [{fold['time_s']['ci_95']['low']:.3f}, {fold['time_s']['ci_95']['high']:.3f}] |
| Mean peak RSS (GiB) | {seq['peak_rss_gb']['mean']:.3f} | {fold['peak_rss_gb']['mean']:.3f} |
| Mean sampled GPU utilization (%) | {seq['avg_gpu_util_percent']['mean']:.3f} | {fold['avg_gpu_util_percent']['mean']:.3f} |
| Mean I/O/compute overlap (s) | {seq['overlap_time_s']['mean']:.3f} | {fold['overlap_time_s']['mean']:.3f} |
| Mean GPU wait time (s) | {seq['gpu_wait_time_s']['mean']:.3f} | {fold['gpu_wait_time_s']['mean']:.3f} |

Paired mean speedup ratio: **{effect['speedup_ratio']['mean']:.4f}x** (95% bootstrap CI [{speed_ci['low']:.4f}, {speed_ci['high']:.4f}]).

Paired mean time saved: **{effect['time_saved_s']['mean']:.3f} s** (95% bootstrap CI [{saved_ci['low']:.3f}, {saved_ci['high']:.3f}]). FoldPipe was faster in {effect['foldpipe_faster_fraction']:.0%} of pairs.

{interpretation}

The JSON artifact contains every raw paired duration and per-shard download, deserialization, training, payload-byte, overlap, and wait-time trace.
"""

output_root = Path("/kaggle/working")
shutil.copy2(results_path, output_root / "benchmark_stats_md17.json")
shutil.copy2(plot_path, output_root / "benchmark_comparison_md17.png")
shutil.copy2("source_manifest.json", output_root / "source_manifest.json")
(output_root / "benchmark_report_md17.md").write_text(report, encoding="utf-8")

print(report)
